# 📄 Smart Document Analyst — CNN Training on Colab
**Team:** Benmouma Salma, Gassi Oumaima
**UIR S8** — AI & Big Data 2025–2026 | Prof. Hakim Hafidi

## 1️⃣ Setup
⚠️ Run cell 1.1, then **restart runtime** (Runtime → Restart session), then continue from 1.2

In [6]:
# 1.1 — Install (RUN FIRST, then restart runtime)
!git clone https://github.com/OumaimaGassi/smart-document-analyst.git
%cd smart-document-analyst

# Fix version conflicts: pin numpy<2, upgrade datasets+huggingface_hub
!pip install -q "numpy<2.0.0"
!pip install -q --upgrade datasets huggingface_hub
!pip install -q crewai crewai-tools google-generativeai pymupdf pytesseract fpdf2 python-dotenv tqdm seaborn
# Re-pin numpy after crewai may have upgraded it
!pip install -q "numpy<2.0.0"

print('\n' + '='*60)
print('⚠️  NOW: Runtime → Restart session')
print('   Then skip this cell, run from 1.2')
print('='*60)

fatal: destination path 'smart-document-analyst' already exists and is not an empty directory.
/content/smart-document-analyst/smart-document-analyst
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 12.6 MB/s eta 0:00:00

⚠️  NOW: Runtime → Restart session
   Then skip this cell, run from 1.2


In [1]:
# 1.2 — After restart, verify everything works
%cd /content/smart-document-analyst
import numpy as np, torch
print(f'NumPy:   {np.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU:     {torch.cuda.get_device_name(0)}')
from sklearn.metrics import accuracy_score
from datasets import load_dataset
print('\n✅ All imports OK!')

/content/smart-document-analyst
NumPy:   1.26.4
PyTorch: 2.10.0+cu128
CUDA:    True
GPU:     Tesla T4

✅ All imports OK!


## 2️⃣ Download RVL-CDIP Dataset Subset

In [2]:
import os
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

ALL_CLASSES = ['letter','memo','email','file_folder','form','handwritten',
    'invoice','advertisement','budget','news_article','presentation',
    'scientific_publication','questionnaire','resume','scientific_report','specification']
SUBSET_CLASSES = ['letter','form','invoice','resume','scientific_publication','email','memo','advertisement']
SUBSET_INDICES = [ALL_CLASSES.index(c) for c in SUBSET_CLASSES]
IMAGES_PER_CLASS = 2000
print(f'Classes: {SUBSET_CLASSES}\nImages/class: {IMAGES_PER_CLASS}')

Classes: ['letter', 'form', 'invoice', 'resume', 'scientific_publication', 'email', 'memo', 'advertisement']
Images/class: 2000


In [ ]:
# FIX 1: use the Parquet-based mirror — 'aharley/rvl_cdip' now has a
# loading script (rvl_cdip.py) which is no longer supported by datasets>=3.
# 'rvl-cdip' (no underscore, different org) is the standard Parquet version.
# FIX 2: removed trust_remote_code (no longer accepted) and
#         added verification_mode='no_checks' to skip slow checksum step.
print('Loading RVL-CDIP from HuggingFace (Parquet mirror)...')
dataset = load_dataset(
    'rvl-cdip',
    split='train',
    streaming=True,
    verification_mode='no_checks'
)

base_dir = 'data/dataset'
for c in SUBSET_CLASSES: os.makedirs(f'{base_dir}/{c}', exist_ok=True)

counts = {c: 0 for c in SUBSET_CLASSES}
saved = 0
for sample in tqdm(dataset, desc='Downloading'):
    idx = sample['label']
    if idx not in SUBSET_INDICES: continue
    cls = ALL_CLASSES[idx]
    if counts[cls] >= IMAGES_PER_CLASS: continue

    # FIX 3: in streaming mode 'image' can arrive as a raw dict
    # {'bytes': ..., 'path': ...} instead of a decoded PIL Image.
    img = sample['image']
    if isinstance(img, dict):
        from io import BytesIO
        img = Image.open(BytesIO(img['bytes']))
    if img.mode != 'RGB':
        img = img.convert('RGB')

    # FIX 4: save as JPEG (quality=85) instead of PNG —
    # ~5x faster saves and ~70% less disk usage on Colab.
    img.save(f'{base_dir}/{cls}/{cls}_{counts[cls]:04d}.jpg', quality=85)
    counts[cls] += 1; saved += 1
    if all(v >= IMAGES_PER_CLASS for v in counts.values()): break

print(f'\n✅ {saved} images downloaded')
for c, n in counts.items(): print(f'   {c}: {n}')

## 3️⃣ Train CNN

In [ ]:
import sys, time, random, json
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
from datetime import datetime

sys.path.insert(0, '.')
from src.models.document_classifier import DocumentClassifierCNN
from src.utils.preprocessing import DOCUMENT_CLASSES_SUBSET

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train_tf = transforms.Compose([transforms.Resize((256,256)), transforms.RandomCrop(224),
    transforms.Grayscale(3), transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
    transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = transforms.Compose([transforms.Resize((224,224)), transforms.Grayscale(3),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# FIX 5: build three separate ImageFolder instances so val and test
# get val_tf (no augmentation) instead of train_tf.
# Previously all splits shared the same 'ds' with train_tf,
# causing augmented validation — inflated/noisy val metrics.
train_ds_full = datasets.ImageFolder('data/dataset', transform=train_tf)
val_ds_full   = datasets.ImageFolder('data/dataset', transform=val_tf)
test_ds_full  = datasets.ImageFolder('data/dataset', transform=val_tf)
print(f'Total: {len(train_ds_full)} | Classes: {train_ds_full.classes}')

n = len(train_ds_full)
nt, nv = int(0.8*n), int(0.1*n); ne = n - nt - nv
idx = list(range(n))
random.seed(SEED); random.shuffle(idx)
train_s = Subset(train_ds_full, idx[:nt])
val_s   = Subset(val_ds_full,   idx[nt:nt+nv])
test_s  = Subset(test_ds_full,  idx[nt+nv:])
print(f'Train: {nt} | Val: {nv} | Test: {ne}')

train_dl = DataLoader(train_s, 32, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_s,   32, num_workers=2)
test_dl  = DataLoader(test_s,  32, num_workers=2)

In [ ]:
NC = len(DOCUMENT_CLASSES_SUBSET)
model = DocumentClassifierCNN(NC, True, 0.3).to(device)
print(f'Trainable params: {model.get_trainable_params():,}')
crit = nn.CrossEntropyLoss()
opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, 20, 1e-6)

In [ ]:
EPOCHS, PAT = 20, 5
best, pc = 0.0, 0
H = {'tl':[],'ta':[],'vl':[],'va':[]}
for ep in range(EPOCHS):
    model.train(); tl=tc=tt=0
    for x,y in tqdm(train_dl, desc=f'E{ep+1}/{EPOCHS}', leave=False):
        x,y=x.to(device),y.to(device); opt.zero_grad()
        o=model(x); l=crit(o,y); l.backward(); opt.step()
        tl+=l.item()*x.size(0); _,p=o.max(1); tt+=y.size(0); tc+=p.eq(y).sum().item()
    model.eval(); vl=vc=vt=0
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.to(device),y.to(device); o=model(x); l=crit(o,y)
            vl+=l.item()*x.size(0); _,p=o.max(1); vt+=y.size(0); vc+=p.eq(y).sum().item()
    ta,va=tc/tt,vc/vt; H['tl'].append(tl/tt); H['ta'].append(ta); H['vl'].append(vl/vt); H['va'].append(va)
    sched.step()
    print(f'E{ep+1:2d} | Train {tl/tt:.4f}/{ta:.4f} | Val {vl/vt:.4f}/{va:.4f}')
    if va>best: best=va; pc=0; model.save_model('model/document_classifier.pt'); print(f'  💾 Saved ({va:.4f})')
    else:
        pc+=1
        if pc>=PAT: print(f'Early stop E{ep+1}'); break
print(f'\n✅ Best: {best:.4f}')

## 4️⃣ Evaluation

In [ ]:
# FIX 6: explicitly move the loaded model to device before eval.
# load_model returns the model on CPU regardless of where it was saved;
# without .to(device) the forward pass crashes when inputs are on GPU.
bm = DocumentClassifierCNN.load_model('model/document_classifier.pt', NC, str(device))
bm = bm.to(device)
bm.eval()
ap,al = [],[]
with torch.no_grad():
    for x,y in tqdm(test_dl, desc='Test'):
        _,p = bm(x.to(device)).max(1); ap.extend(p.cpu().numpy()); al.extend(y.numpy())
print('\n📊 CLASSIFICATION REPORT')
print(classification_report(al, ap, target_names=DOCUMENT_CLASSES_SUBSET))
print(f'Accuracy: {accuracy_score(al, ap):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
cm = confusion_matrix(al, ap)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=DOCUMENT_CLASSES_SUBSET, yticklabels=DOCUMENT_CLASSES_SUBSET)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True'); axes[0].set_title('Confusion Matrix')
ep = range(1, len(H['tl'])+1)
axes[1].plot(ep,H['tl'],'b-o',ms=3,label='Train'); axes[1].plot(ep,H['vl'],'r-o',ms=3,label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(ep,H['ta'],'b-o',ms=3,label='Train'); axes[2].plot(ep,H['va'],'r-o',ms=3,label='Val')
axes[2].set_title('Accuracy'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle('Document Classifier — Benmouma Salma & Gassi Oumaima', fontsize=14)
plt.tight_layout(); os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/evaluation_results.png', dpi=150); plt.show()

In [ ]:
meta = {'timestamp': datetime.now().isoformat(), 'team': 'Benmouma Salma, Gassi Oumaima',
    'accuracy': float(accuracy_score(al,ap)), 'best_val_acc': float(best),
    'num_classes': NC, 'classes': DOCUMENT_CLASSES_SUBSET,
    'epochs': len(H['tl']), 'device': str(device), 'pytorch': torch.__version__}
with open('model/training_metadata.json','w') as f: json.dump(meta,f,indent=2)
print('✅ Metadata saved')

## 5️⃣ Push to GitHub

In [ ]:
from getpass import getpass
token = getpass('GitHub Token: ')
!git remote set-url origin https://{token}@github.com/OumaimaGassi/smart-document-analyst.git
!git config user.name "OumaimaGassi"
!git config user.email "oumaima.gassi@uir.ac.ma"
!git add model/ outputs/
!git commit -m "feat: trained CNN model (ResNet-18, RVL-CDIP)"
!git push origin main
print('\n✅ Pushed!')

## ✅ Done! Clone anywhere:
```bash
git clone https://github.com/OumaimaGassi/smart-document-analyst.git
cd smart-document-analyst && pip install -r requirements.txt
python src/main.py --input document.pdf --hitl
```